# The `normal` function

The function `normal` from the module `modelnormalize` **normalizes** model equations: it takes a business-logic equation written as `lhs = rhs` and rewrites it so that the equation's endogenous variable stands alone on the left-hand side. In a model each equation determines one endogenous variable, so the normalized form of equation $i$ is $y_i = F_i(y, x)$ — the right-hand side may contain the other endogenous variables of the model and any lagged (or leaded) values, including lags of $y_i$ itself; only the current-period $y_i$ must not appear on the right. This is the form the ModelFlow solvers require.

Beyond isolating the endogenous variable, `normal` can:

- **preprocess** EViews-style transformation functions (`DLOG`, `DIFF`, `PCT_GROWTH`, `PCY`, `MOVAVG`, `LOGIT`, `D`, …) into elementary expressions,
- **invert** a transformation applied to the endogenous variable on the left-hand side (for instance solve `dlog(y) = rhs` for `y`),
- inject an **add factor** (adjustment term) `<var>_A` together with the equation that calculates it from data,
- make the equation **fixable** (exogenizable) through a dummy `<var>_D` and a fixing value `<var>_X`,
- produce a **fitted** version of the equation, stripped of add factor and fixing terms,
- recast **implicit** equations (equilibrium conditions) as residual equations for the solver.

`normal` is the workhorse behind `Makemodel` and therefore also the `%%makemymodel` cell magic. It returns an instance of the class `Normalized_frml`, which holds the different variants of the equation.

This notebook walks through the main use cases. The reader is assumed to be comfortable with Python and with macroeconomic model equations.

## Imports

Import the `normal` function and the `Normalized_frml` result class.

In [2]:
from modelnormalize import normal,Normalized_frml

## The result class: `Normalized_frml`

Every call to `normal` returns a `Normalized_frml` instance. Its attributes hold the different representations of the equation — original, preprocessed, normalized, the add-factor calculation, the fitted version and so on. Displaying the instance prints the attributes that are set.

In [3]:
print(Normalized_frml.__doc__)


Class defining the result from normalization of an expression.

Attributes:
    endo_var (str): The endogenous variable.
    eviews (str): The EViews representation of the expression.
    original (str): The original Business logic expression.
    preprocessed (str): The preprocessed expression.
    normalized (str): The normalized expression.
    calc_add_factor (str): The calculated additional factor.
    un_normalized (str): The un-normalized expression.
    fitted (str): The fitted expression.



## The signature of `normal`

The docstring below lists all arguments. The ones used in this notebook are `the_endo`, `add_add_factor`, `add_suffix`, `endo_lhs`, `make_fixable`, `make_fitted` and `implicit`.

In [4]:
print(normal.__doc__)


normalize an expression g(y,x) = f(y,x) ==> y = F(x,z)

Default find the expression for the first variable on the left hand side (lhs)

The variable - without lags-  should not be on rhs. 

Args:
    ind_o (str): input expression, no $ and no frml name just lhs=rhs
    the_endo (str, optional): the endogeneous to isolate on the left hans side. if the first variable in the lhs. 
                              It shoud be on the left hand side. 
    add_add_factor (bool, optional): force introduction aof adjustment term, and an expression to calculate it
    do_preprocess (bool, optional): DESCRIPTION. preprocess the expression
    endo_lhs (bool, optional): If false, accept to normalize for a rhs endogeneous variable 
    make_fixable  (bool, optional): also make this equation exogenizable  
    fitted (bool,optional) : create a fitted equations, without exo and adjustment 
    implicit (bool,optional) : This is an implicit frml so to transform to endo__res = (lhs)-(rhs) 
    
preproces

Autoreload keeps the imported module in sync with the source file — convenient while developing `modelnormalize`, not needed if you just use the function.

In [5]:
%load_ext autoreload
%autoreload 2

## Basic normalization

The simplest possible case: `a = b`. Variable names are upper-cased, and by default an **add factor** `A_A` is introduced. `Calc_add_factor` shows the expression used to compute the add factor from historical data, so that the equation tracks the data exactly.

In [6]:
normal('a=b')

Endo_var        : A
Original        : a=b
Preprocessed    : A=B
Normalized      : A = (B + A_A)                               
Calc_add_factor : A_A = (A) - (B)

With `add_add_factor=0` no adjustment term is introduced — the result is the plain normalized equation.

In [7]:
normal('a=b',add_add_factor=0)

Endo_var        : A
Original        : a=b
Preprocessed    : A=B
Normalized      : A = B

In [8]:
normal('a-c=b',add_add_factor=0)

Endo_var        : A
Original        : a-c=b
Preprocessed    : A-C=B
Normalized      : A = C+ (B) 

## Add factor, fixing and fitted equation combined

All the bells and whistles at once:

- `add_suffix='_AERR'` gives the add factor a custom name (`A_AERR` instead of the default `A_A`),
- `make_fixable=True` makes the equation exogenizable: the dummy `A_D` and the fixing value `A_X` are introduced, so setting `A_D = 1` fixes `A` at the value of `A_X`,
- `make_fitted=True` creates the fitted equation `A_FITTED` — the pure right-hand side without add factor and fixing terms.

In [9]:
normal('a=b',add_add_factor=1,add_suffix= '_AERR',make_fitted=True,make_fixable=True )

Endo_var        : A
Original        : a=b
Preprocessed    : A=B
Normalized      : A = (B + A_AERR)* (1-A_D)+ A_X*A_D 
Calc_add_factor : A_AERR = (A) - (B)
Fitted          : A_FITTED = B

## Built-in transformation functions on the right-hand side

The preprocessing step unfolds EViews-style transformation functions into elementary expressions. First `dlog(b)` — the change in the logarithm — which becomes `log(B) - log(B(-1))`.

In [10]:
normal('a=dlog(b)',add_add_factor=1,add_suffix= '_AERR',make_fitted=True,make_fixable=True )

Endo_var        : A
Original        : a=dlog(b)
Preprocessed    : A=((LOG(B))-(LOG(B(-1))))
Normalized      : A = (((log(B))-(log(B(-1)))) + A_AERR)* (1-A_D)+ A_X*A_D 
Calc_add_factor : A_AERR = (A) - (((log(B))-(log(B(-1)))))
Fitted          : A_FITTED = ((LOG(B))-(LOG(B(-1))))

`logit(b)` is unfolded to `log(B/(1 - B))` — useful when a variable must stay between 0 and 1.

In [11]:
normal('a=logit(b)',add_add_factor=1,add_suffix= '_AERR',make_fitted=True,make_fixable=True )

Endo_var        : A
Original        : a=logit(b)
Preprocessed    : A=(LOG(B/(1.0 -B)))
Normalized      : A = ((log(B/(1.0 -B))) + A_AERR)* (1-A_D)+ A_X*A_D 
Calc_add_factor : A_AERR = (A) - ((log(B/(1.0 -B))))
Fitted          : A_FITTED = (LOG(B/(1.0 -B)))

`PCT_GROWTH(b)` is the percentage growth from the previous period: `100*(B/B(-1) - 1)`.

In [12]:
normal('a=PCT_GROWTH(b)',add_add_factor=0)

Endo_var        : A
Original        : a=PCT_GROWTH(b)
Preprocessed    : A= (100 * ( (B) / (B(-1)) -1)) 
Normalized      : A =  (100 * ( (B) / (B(-1)) -1))

`PCY(b)` is the year-over-year percentage growth — the change over four periods, so it assumes quarterly data.

In [13]:
normal('a=PCY(b)',add_add_factor=1,add_suffix= '_AERR',make_fitted=True,make_fixable=True )

Endo_var        : A
Original        : a=PCY(b)
Preprocessed    : A= (100 * ( (B) / (B(-4)) -1)) 
Normalized      : A = ( (100 * ( (B) / (B(-4)) -1)) + A_AERR)* (1-A_D)+ A_X*A_D 
Calc_add_factor : A_AERR = (A) - ( (100 * ( (B) / (B(-4)) -1)))
Fitted          : A_FITTED =  (100 * ( (B) / (B(-4)) -1))

`DIFF(b)` is the first difference `B - B(-1)`.

In [14]:
normal('a=DIFF(b)',add_add_factor=1,add_suffix= '_AERR',make_fitted=True,make_fixable=True )

Endo_var        : A
Original        : a=DIFF(b)
Preprocessed    : A=((B)-(B(-1)))
Normalized      : A = (((B)-(B(-1))) + A_AERR)* (1-A_D)+ A_X*A_D 
Calc_add_factor : A_AERR = (A) - (((B)-(B(-1))))
Fitted          : A_FITTED = ((B)-(B(-1)))

`MOVAVG(b,3)` is the moving average over the current and the two preceding periods.

In [15]:
normal('a=MOVAVG(b,3)',add_add_factor=1,add_suffix= '_AERR',make_fitted=True,make_fixable=True )

Endo_var        : A
Original        : a=MOVAVG(b,3)
Preprocessed    : A=((B+B(-1)+B(-2))/3.0)
Normalized      : A = (((B+B(-1)+B(-2))/3.0) + A_AERR)* (1-A_D)+ A_X*A_D 
Calc_add_factor : A_AERR = (A) - (((B+B(-1)+B(-2))/3.0))
Fitted          : A_FITTED = ((B+B(-1)+B(-2))/3.0)

## Transformations on the left-hand side

When the endogenous variable enters through a transformation on the left-hand side, `normal` **inverts** the transformation. Here `PCT_growth(a) = n(-1)` is solved for the level of `A`.

In [16]:
normal('PCT_growth(a) = n(-1)',add_add_factor=0)

Endo_var        : A
Original        : PCT_growth(a) = n(-1)
Preprocessed    :  (100 * ( (A) / (A(-1)) -1)) =N(-1)
Normalized      : A =  (N(-1)) *A(-1)/100+A(-1)

Transformation functions can be nested — here a moving average of the percentage change `pct(b)`. Note how the lag operator is distributed over the arguments when the expression is unfolded.

In [17]:
normal('a = movavg(pct(b),2)',add_add_factor=0)

Endo_var        : A
Original        : a = movavg(pct(b),2)
Preprocessed    : A=(( (100 * ( (B) / (B(-1)) -1)) + (100 * ( (B(-1)) / (B(-2)) -1)) )/2.0)
Normalized      : A = (( (100 * ( (B) / (B(-1)) -1)) + (100 * ( (B(-1)) / (B(-2)) -1)) )/2.0)

`pct_growth` on **both** sides: the left-hand side is inverted to isolate `C`, while the right-hand side is simply unfolded.

Note: the right-hand variable is called `DD` because `D` is reserved for the EViews difference operator `D(...)` — once the equation is unfolded, a lag like `D(-1)` would be indistinguishable from a call to the operator, so `normal` raises an error asking you to rename such a variable.

In [18]:
normal('pct_growth(c) = pct_growth(dd)',add_add_factor=0)

Endo_var        : C
Original        : pct_growth(c) = pct_growth(dd)
Preprocessed    :  (100 * ( (C) / (C(-1)) -1)) = (100 * ( (DD) / (DD(-1)) -1)) 
Normalized      : C =  ((100 * ( (DD) / (DD(-1)) -1))) *C(-1)/100+C(-1)

A variable can not share its name with one of the reserved function names (`D`, `DIFF`, `DLOG`, `PCT_GROWTH`, `PCY`, `LOGIT`, `MOVAVG`): as soon as such a variable appears with a lag — like `DIFF(-1)` — it is indistinguishable from a function call, and `normal` raises an explicit error asking you to rename it. In the two cells below the exception is caught just to display the message.

First a variable named `DIFF`, lagged directly by the user:

In [19]:
try:
    normal('a = diff(b) + diff(-1)',add_add_factor=0)
except Exception as e: 
    print('Exception raised')
    print(e)

Exception raised
A variable can not be named DIFF, as DIFF( is a function in the business language.
DIFF(-1) looks like a lagged variable DIFF. Rename the variable.
Original expression: a=diff(b)+diff(-1)
Processed so far   : A=((B)-(B(-1)))+DIFF(-1)


And a variable named `D` — here the offending lag `D(-1)` is not written by the user but *created* by the unfolding of `pct_growth(d)`, as the "Processed so far" line shows:

In [20]:
try:
    normal('pct_growth(c) = pct_growth(d)',add_add_factor=0)
except Exception as e: 
    print('Exception raised')
    print(e)

Exception raised
A variable can not be named D, as D( is a function in the business language.
D(-1) looks like a lagged variable D. Rename the variable.
Original expression: pct_growth(c)=pct_growth(d)
Processed so far   :  (100 * ( (C) / (C(-1)) -1)) = (100 * ( (D) / (D(-1)) -1)) 


When the left-hand side is a transformation and an add factor is requested (the default), the add factor enters in the **transformed space**: here `C_A` is measured in percentage growth, and `Calc_add_factor` computes it accordingly.

In [21]:
normal('pct_growth(c) = z+pct(b) + pct(e)')

Endo_var        : C
Original        : pct_growth(c) = z+pct(b) + pct(e)
Preprocessed    :  (100 * ( (C) / (C(-1)) -1)) =Z+ (100 * ( (B) / (B(-1)) -1)) + (100 * ( (E) / (E(-1)) -1)) 
Normalized      : C = C_A*C(-1)/100+ (Z+ (100 * ( (B) / (B(-1)) -1)) + (100 * ( (E) / (E(-1)) -1))) *C(-1)/100+C(-1)
Calc_add_factor : C_A = 100*C/C(-1)- ((Z+ (100 * ( (B) / (B(-1)) -1)) + (100 * ( (E) / (E(-1)) -1)))) -100

## A real-world example

An estimated error-correction equation from a World Bank EViews model. `DLOG` on the left-hand side gives a **multiplicative** normalization through `EXP(...)`, and the add factor works on the log change. EViews-specific pieces such as `@DURING("2011")` pass through the preprocessing. The `.fprint` attribute prints the result without truncation — handy for long equations.

In [22]:
normal("DLOG(SAUNECONGOVTXN) = -0.323583422052*(LOG(SAUNECONGOVTXN(-1))-GOVSHAREWB*LOG(SAUNEYWRPGOVCN(-1))-(1-GOVSHAREWB)*LOG(SAUNECONPRVTXN(-1)))+0.545415878897*DLOG(SAUNECONGOVTXN(-1))+(1-0.545415878897)*(GOVSHAREWB)*DLOG(SAUNEYWRPGOVCN) +(1-0.545415878897)*(1-GOVSHAREWB)*DLOG(SAUNECONPRVTXN)-1.56254616684-0.0613991001064*@DURING(""2011"")").fprint


Endo_var        : SAUNECONGOVTXN
Original        : DLOG(SAUNECONGOVTXN) = -0.323583422052*(LOG(SAUNECONGOVTXN(-1))-GOVSHAREWB*LOG(SAUNEYWRPGOVCN(-1))-(1-GOVSHAREWB)*LOG(SAUNECONPRVTXN(-1)))+0.545415878897*DLOG(SAUNECONGOVTXN(-1))+(1-0.545415878897)*(GOVSHAREWB)*DLOG(SAUNEYWRPGOVCN) +(1-0.545415878897)*(1-GOVSHAREWB)*DLOG(SAUNECONPRVTXN)-1.56254616684-0.0613991001064*@DURING(2011)
Preprocessed    : ((LOG(SAUNECONGOVTXN))-(LOG(SAUNECONGOVTXN(-1))))=-0.323583422052*(LOG(SAUNECONGOVTXN(-1))-GOVSHAREWB*LOG(SAUNEYWRPGOVCN(-1))-(1-GOVSHAREWB)*LOG(SAUNECONPRVTXN(-1)))+0.545415878897*((LOG(SAUNECONGOVTXN(-1)))-(LOG(SAUNECONGOVTXN(-2))))+(1-0.545415878897)*(GOVSHAREWB)*((LOG(SAUNEYWRPGOVCN))-(LOG(SAUNEYWRPGOVCN(-1))))+(1-0.545415878897)*(1-GOVSHAREWB)*((LOG(SAUNECONPRVTXN))-(LOG(SAUNECONPRVTXN(-1))))-1.56254616684-0.0613991001064*@DURING(+2011)
Normalized      : SAUNECONGOVTXN = SAUNECONGOVTXN(-1)*EXP(SAUNECONGOVTXN_A+ (-0.323583422052*(LOG(SAUNECONGOVTXN(-1))-GOVSHAREWB*LOG(SAUNEYWRPGOVCN(-1))

`DIFF` on the left-hand side is inverted additively: `A = rhs + A(-1)`.

In [23]:
normal("Diff(a) = b")

Endo_var        : A
Original        : Diff(a) = b
Preprocessed    : ((A)-(A(-1)))=B
Normalized      : A = A_A+ (B) +A(-1)
Calc_add_factor : A_A = A- ((B)) -A(-1)

## Leads and the EViews `D()` operator

The EViews difference operator `D(x, n, s)` is also handled, and expressions may contain **leads** such as `QLHP(+1)`.

In [24]:
normal('a = D( LOG(QLHP(+1)), 0, 1 )')

Endo_var        : A
Original        : a = D( LOG(QLHP(+1)), 0, 1 )
Preprocessed    : A=((LOG(QLHP(+1)))-(LOG(QLHP)))
Normalized      : A = (((log(QLHP(+1)))-(log(QLHP))) + A_A)                               
Calc_add_factor : A_A = (A) - (((log(QLHP(+1)))-(log(QLHP))))

`D(x)` with default arguments is a plain first difference — the result is the same as above.

In [25]:
normal('a = D( LOG(QLHP(+1)))')

Endo_var        : A
Original        : a = D( LOG(QLHP(+1)))
Preprocessed    : A=((LOG(QLHP(+1)))-(LOG(QLHP)))
Normalized      : A = (((log(QLHP(+1)))-(log(QLHP))) + A_A)                               
Calc_add_factor : A_A = (A) - (((log(QLHP(+1)))-(log(QLHP))))

## Choosing the endogenous variable

By default the first variable on the left-hand side becomes the endogenous variable. With `the_endo` and `endo_lhs=False` the equation can instead be solved for a variable on the **right-hand side** — here `F`. Combined with `make_fixable=True` the resulting equation is also exogenizable.

In [26]:
normal('a = gamma+ f+O',the_endo='f',endo_lhs=False,make_fixable =True)

Endo_var        : F
Original        : a = gamma+ f+O
Preprocessed    : A=GAMMA+F+O
Normalized      : F = (A-F_A-GAMMA-O) * (1-F_D)+ F_X*F_D 
Calc_add_factor : F_A = A-F-GAMMA-O

## Implicit equations

Some equations — typically equilibrium conditions — cannot be solved analytically for the endogenous variable. With `implicit=True` the equation is recast as a **residual equation**: `PRICE___RES = supply − demand`, which is driven to zero by adjusting the variable named in `the_endo` (here `PRICE`).

Note that a model containing implicit equations has to be solved with one of the **Newton-type solvers** — the Gauss-Seidel type solvers require every equation to be normalized with the endogenous variable alone on the left-hand side, which is exactly what an implicit equation lacks.

In [27]:
normal('demand = supply',the_endo='price',implicit=True)

Endo_var        : PRICE
Original        : demand = supply
Preprocessed    : DEMAND=SUPPLY
Normalized      : PRICE___RES = ( SUPPLY ) - ( DEMAND )
Un_normalized   : PRICE___RES = ( SUPPLY ) - ( DEMAND )

## Summary: the preprocessing steps

The table lists the preprocessing steps in the order they are applied. Each rewrite is repeated until no occurrence is left, so the functions can be nested — `movavg(pct(b),2)` for instance is unfolded completely.

| Step | Pattern | Rewritten as | Meaning |
|---|---|---|---|
| 1 | *whole equation* | upper-cased, blanks removed | housekeeping |
| 2 | `PCT(x)` | `PCT_GROWTH(x)` | alias — handled in step 6 |
| 3 | `DLOG(x)` | `DIFF(LOG(x))` | change in the logarithm — handled further in step 8 |
| 4 | `LOGIT(x)` | `(LOG(x/(1.0 - x)))` | log-odds of a variable between 0 and 1 |
| 5 | `MOVAVG(x,n)` | `((x + x(-1) + … + x(-n+1))/n)` | moving average over `n` periods |
| 6 | `PCT_GROWTH(x)` | `(100*((x)/(x(-1)) - 1))` | percentage growth from the previous period |
| 7 | `PCY(x)` | `(100*((x)/(x(-4)) - 1))` | year-over-year percentage growth (quarterly data) |
| 8 | `DIFF(x)` | `((x) - (x(-1)))` | first difference |
| 9 | `D(x)` or `D(x,0,1)` | `((x) - (x(-1)))` | EViews difference operator |

Before every rewrite the argument is checked: a pure signed integer like `DIFF(-1)` must be a *lagged variable* named like a function, and an explicit error is raised (see the section on reserved names above).